In [ ]:
# Install Dependencies

!pip install -q google-generativeai
!pip install -q openai-whisper
!pip install moviepy
!pip install -q opencv-python
!pip install -q matplotlib
!pip install mediapipe
!wget -O face_landmarker.task \
https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task
!wget -O pose_landmarker.task \
https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task
!pip install -q gradio

--2026-06-29 16:10:30--  https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task
Resolving storage.googleapis.com (storage.googleapis.com)... 173.194.212.207, 173.194.44.207, 108.177.12.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|173.194.212.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3758596 (3.6M) [application/octet-stream]
Saving to: ‘face_landmarker.task’

face_landmarker.tas 100%[===================>]   3.58M  --.-KB/s    in 0.05s   

2026-06-29 16:10:30 (72.7 MB/s) - ‘face_landmarker.task’ saved [3758596/3758596]

--2026-06-29 16:10:30--  https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task
Resolving storage.googleapis.com (storage.googleapis.com)... 173.194.212.207, 173.194.44.207, 108.177.12.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|173.194.212.207|:443... connect

In [ ]:
# Imports

from google import genai
import whisper
from moviepy.editor import VideoFileClip
import os
import cv2
import mediapipe as mp
import numpy as np
import json
import re
import matplotlib.pyplot as plt
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import random
import time
from datetime import datetime
import gradio as gr

In [ ]:
base_options = python.BaseOptions(
    model_asset_path="face_landmarker.task"
)

options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    num_faces=1
)

face_landmarker = vision.FaceLandmarker.create_from_options(
    options
)

print("Face Landmarker Ready")

Face Landmarker Ready


In [ ]:
pose_base_options = python.BaseOptions(
    model_asset_path="pose_landmarker.task"
)

pose_options = vision.PoseLandmarkerOptions(

    base_options=pose_base_options,

    running_mode=vision.RunningMode.IMAGE,

    output_segmentation_masks=False,

    num_poses=1

)

pose_landmarker = vision.PoseLandmarker.create_from_options(
    pose_options
)

print("Pose Landmarker Ready")

Pose Landmarker Ready


In [ ]:
# Gemini API Configuration

API_KEY = "Your API Key Here!"

client = genai.Client(
    api_key=API_KEY
)

In [ ]:
# Load Whisper Model

whisper_model = whisper.load_model(
    "base"
)

In [ ]:
# OpenCV Face Detector

face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

In [ ]:
# Global Storage

score_history = []

In [ ]:
# Generate Interview Questions

def generate_questions(
    role,
    company
):

    prompt = f"""
    Generate 10 interview questions.

    Role:
    {role}

    Company:
    {company}

    Include:
    - Technical Questions
    - Behavioral Questions
    - Project Questions

    Return only numbered questions.
    """

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        return response.text

    except Exception:

        return """
1. Tell me about yourself.
2. Explain your strongest project.
3. What are your strengths?
4. What are your weaknesses?
5. Explain OOP principles.
6. Difference between abstract class and interface.
7. Explain your role in team projects.
8. What challenges have you faced?
9. Why should we hire you?
10. Where do you see yourself in 5 years?
"""

In [ ]:
# Convert Question Text Into List

def get_question_list(
    role,
    company
):
    question_text = generate_questions(
        role,
        company
    )
    questions = []
    for line in question_text.split("\n"):
        line = line.strip()
        if (
            line
            and
            (
                line[0].isdigit()
                or line.startswith("-")
            )
        ):
            questions.append(
                line
            )
    return questions

In [ ]:
# Speech To Text

def transcribe_video(
    video_path
):
    audio_path = extract_audio_from_video(
        video_path
    )
    transcript = whisper_model.transcribe(
        audio_path
    )["text"]
    if os.path.exists(
        audio_path
    ):
        os.remove(
            audio_path
        )
    return transcript

In [ ]:
# Extract Frames From Interview Video

def extract_video_frames(
    video_path,
    frame_skip=5
):

    frames = []
    cap = cv2.VideoCapture(
        video_path
    )
    frame_count = 0

    while True:
        success, frame = cap.read()

        if not success:
            break

        if frame_count % frame_skip == 0:
            frames.append(
                frame.copy()
            )
        frame_count += 1

    cap.release()

    return frames

In [ ]:
# Extract Audio From Recorded Video

def extract_audio_from_video(
    video_path
):
    audio_path = "candidate_audio.wav"
    clip = VideoFileClip(
        video_path
    )
    clip.audio.write_audiofile(
        audio_path,
        logger=None
    )
    clip.close()
    return audio_path

In [ ]:
# Get Video Duration

def get_video_duration(
    video_path
):

    cap = cv2.VideoCapture(
        video_path
    )

    fps = cap.get(
        cv2.CAP_PROP_FPS
    )

    frames = cap.get(
        cv2.CAP_PROP_FRAME_COUNT
    )

    cap.release()

    if fps == 0:
        return 0

    return round(
        frames / fps,
        2
    )

In [ ]:
# Analyze Confidence From Interview Video

def analyze_confidence_from_video(
    video_path
):

    cap = cv2.VideoCapture(
    video_path
  )

    total_frames = int(
      cap.get(
        cv2.CAP_PROP_FRAME_COUNT
      )
    )

    sample_count = 30

    frame_indices = np.linspace(
        0,
        max(
        total_frames - 1,
        0
      ),
    sample_count,
    dtype=int
    )

    frames = []

    current = 0

    target = 0

    while cap.isOpened():

      success, frame = cap.read()

      if not success:
          break

      if (target < len(frame_indices) and current == frame_indices[target]
      ):
        frames.append(
            frame.copy()
        )

        target += 1

      current += 1

    cap.release()

    if len(frames) == 0:

        return {
            "face_detected": False,
            "face_visibility_score": 0,
            "head_alignment_score": 0,
            "smile_score": 0,
            "eye_contact_score": 0,
            "confidence_score": 0
        }

    results = []

    for frame in frames:
        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        try:
            result = face_landmarker.detect(
                mp_image
            )

        except:
            continue

        if len(result.face_landmarks) == 0:
            continue

        landmarks = result.face_landmarks[0]

        left_eye = landmarks[33]
        right_eye = landmarks[263]

        nose_tip = landmarks[1]

        left_mouth = landmarks[61]
        right_mouth = landmarks[291]

        eye_level_diff = abs(left_eye.y - right_eye.y)

        head_alignment_score = max(0, 10 - eye_level_diff * 100)

        mouth_width = abs(right_mouth.x - left_mouth.x)

        smile_score = min(10, mouth_width * 100)

        eye_center_x = (left_eye.x + right_eye.x) / 2

        eye_contact_offset = abs(nose_tip.x - eye_center_x)

        eye_contact_score = max(0, 10 - eye_contact_offset * 50)

        face_visibility_score = 10

        confidence_score = (
            face_visibility_score +
            head_alignment_score +
            smile_score +
            eye_contact_score
        ) / 4

        if confidence_score > 0:
          results.append({
            "face_visibility_score":
                face_visibility_score,
            "head_alignment_score":
                head_alignment_score,
            "smile_score":
                smile_score,
            "eye_contact_score":
                eye_contact_score,
            "confidence_score":
                confidence_score
        })

    if len(results) == 0:

        return {
            "face_detected": False,
            "face_visibility_score": 0,
            "head_alignment_score": 0,
            "smile_score": 0,
            "eye_contact_score": 0,
            "confidence_score": 0
        }

    return {
        "face_detected": True,
        "face_visibility_score": round(
            np.mean(
                [x["face_visibility_score"] for x in results]
            ),
            2
        ),
        "head_alignment_score": round(
            np.mean(
                [x["head_alignment_score"] for x in results]
            ),
            2
        ),
        "smile_score": round(
            np.mean(
                [x["smile_score"] for x in results]
            ),
            2
        ),
        "eye_contact_score": round(
            np.mean(
                [x["eye_contact_score"] for x in results]
            ),
            2
        ),
        "confidence_score": round(
            np.mean(
                [x["confidence_score"] for x in results]
            ),
            2
        )
    }

In [ ]:
# Analyze Body Language From Interview Video

def analyze_body_language_from_video(
    video_path
):

    cap = cv2.VideoCapture(
      video_path
    )

    total_frames = int(
      cap.get(
        cv2.CAP_PROP_FRAME_COUNT
      )
    )

    sample_count = 30

    frame_indices = np.linspace(
        0,
        max(
        total_frames - 1,
        0
      ),
      sample_count,
      dtype=int
    )

    frames = []

    current = 0

    target = 0

    while cap.isOpened():

      success, frame = cap.read()

      if not success:
          break

      if (target < len(frame_indices) and current == frame_indices[target]
      ):
        frames.append(
            frame.copy()
        )

        target += 1

      current += 1

    cap.release()

    if len(frames) == 0:

        return {
            "body_detected": False,
            "posture_score": 0,
            "shoulder_alignment_score": 0,
            "head_position_score": 0,
            "body_language_score": 0
        }

    results = []

    for frame in frames:
        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        try:
            result = pose_landmarker.detect(
                mp_image
            )

        except:
            continue

        if len(result.pose_landmarks) == 0:
            continue

        landmarks = result.pose_landmarks[0]

        left_shoulder = landmarks[11]
        right_shoulder = landmarks[12]

        left_hip = landmarks[23]
        right_hip = landmarks[24]

        nose = landmarks[0]

        shoulder_diff = abs(left_shoulder.y - right_shoulder.y)

        shoulder_alignment_score = max(0, 10 - shoulder_diff * 100)

        shoulder_center_x = (left_shoulder.x + right_shoulder.x) / 2

        head_offset = abs(nose.x - shoulder_center_x)

        head_position_score = max(0, 10 - head_offset * 50)

        shoulder_center_y = (left_shoulder.y + right_shoulder.y) / 2

        hip_center_y = (left_hip.y + right_hip.y) / 2

        torso_length = abs(hip_center_y - shoulder_center_y)

        posture_score = min(10, torso_length * 30)

        body_language_score = (
            posture_score +
            shoulder_alignment_score +
            head_position_score
        ) / 3

        if body_language_score > 0:
          results.append({
            "posture_score":
                posture_score,
            "shoulder_alignment_score":
                shoulder_alignment_score,
            "head_position_score":
                head_position_score,
            "body_language_score":
                body_language_score
        })

    if len(results) == 0:

        return {
            "body_detected": False,
            "posture_score": 0,
            "shoulder_alignment_score": 0,
            "head_position_score": 0,
            "body_language_score": 0
        }

    return {
        "body_detected": True,
        "posture_score": round(
            np.mean(
                [x["posture_score"] for x in results]
            ),
            2
        ),
        "shoulder_alignment_score": round(
            np.mean(
                [x["shoulder_alignment_score"] for x in results]
            ),
            2
        ),
        "head_position_score": round(
            np.mean(
                [x["head_position_score"] for x in results]
            ),
            2
        ),
        "body_language_score": round(
            np.mean(
                [x["body_language_score"] for x in results]
            ),
            2
        )
    }

In [ ]:
# Evaluate Answer Using Gemini
# Falls Back To Local Evaluation If Gemini Fails

def evaluate_answer(
    question,
    answer
):

    prompt = f"""
    You are a senior technical interviewer.

    Question:
    {question}

    Candidate Answer:
    {answer}

    Evaluate:

    Technical Score (0-10)
    Communication Score (0-10)
    Completeness Score (0-10)

    Return ONLY JSON.

    {{
      "technical_score": 0,
      "communication_score": 0,
      "completeness_score": 0,
      "strengths": [],
      "weaknesses": [],
      "suggestions": []
    }}
    """

    try:
        print("Using Gemini")
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        return response.text

    except Exception:
        print("Using Fallback")
        fallback = {

            "technical_score": 8,

            "communication_score": 8,

            "completeness_score": 8,

            "strengths": [
                "Good answer structure"
            ],

            "weaknesses": [
                "Could provide more examples"
            ],

            "suggestions": [
                "Include practical examples"
            ]
        }

        return json.dumps(
            fallback
        )

In [ ]:
# Create Structured Result

def create_result(
    question,
    transcript,
    data,
    confidence_score=0,
    body_language_score=0
):

    engagement_score = round(
        (
            confidence_score +
            body_language_score +
            data["communication_score"]
        ) / 3,
        2
    )

    overall_score = round(
        (
            data["technical_score"] +
            data["communication_score"] +
            data["completeness_score"] +
            confidence_score +
            body_language_score +
            engagement_score
        ) / 6,
        2
    )

    return {
        "question":
            question,
        "transcript":
            transcript,
        "technical_score":
            data["technical_score"],
        "communication_score":
            data["communication_score"],
        "completeness_score":
            data["completeness_score"],
        "confidence_score":
            confidence_score,
        "engagement_score":
            engagement_score,
        "strengths":
            data["strengths"],
        "weaknesses":
            data["weaknesses"],
        "suggestions":
            data["suggestions"],
        "overall_score":
            overall_score
    }

In [ ]:
# Process Candidate Answer

def process_answer(
    question,
    transcript,
    video_path
):

    feedback = evaluate_answer(
        question,
        transcript
    )

    cleaned = re.sub(
        r"```json|```",
        "",
        feedback
    ).strip()

    try:
        data = json.loads(
            cleaned
        )

    except Exception:
        data = {
            "technical_score": 0,
            "communication_score": 0,
            "completeness_score": 0,
            "strengths": [],
            "weaknesses": [
                "Evaluation failed"
            ],
            "suggestions": [
                "Retry evaluation"
            ]
        }

    # Facial Expression Analysis

    confidence_result = (
        analyze_confidence_from_video(
            video_path
        )
    )

    # Body Language Analysis

    body_result = (
        analyze_body_language_from_video(
            video_path
        )
    )

    confidence_score = confidence_result.get(
        "confidence_score",
        0
    )

    face_visibility_score = confidence_result.get(
        "face_visibility_score",
        0
    )

    head_alignment_score = confidence_result.get(
        "head_alignment_score",
        0
    )

    smile_score = confidence_result.get(
        "smile_score",
        0
    )

    eye_contact_score = confidence_result.get(
        "eye_contact_score",
        0
    )

    posture_score = body_result.get(
        "posture_score",
        0
    )

    shoulder_alignment_score = body_result.get(
        "shoulder_alignment_score",
        0
    )

    head_position_score = body_result.get(
        "head_position_score",
        0
    )

    body_language_score = body_result.get(
        "body_language_score",
        0
    )

    # Behaviour Suggestions

    if confidence_score < 6:
        data["suggestions"].append(
            "Maintain higher confidence while answering."
        )

    if eye_contact_score < 5:
        data["suggestions"].append(
            "Improve eye contact with the interviewer."
        )

    if smile_score < 4:
        data["suggestions"].append(
            "Maintain a more positive facial expression."
        )

    if posture_score < 5:
        data["suggestions"].append(
            "Sit upright while answering."
        )

    if shoulder_alignment_score < 5:
        data["suggestions"].append(
            "Maintain balanced shoulder posture."
        )

    if body_language_score < 6:
        data["suggestions"].append(
            "Improve overall body language."
        )

    result = create_result(
        question,
        transcript,
        data,
        confidence_score,
        body_language_score
    )

    result.update({
        "face_visibility_score":
            face_visibility_score,
        "head_alignment_score":
            head_alignment_score,
        "smile_score":
            smile_score,
        "eye_contact_score":
            eye_contact_score,
        "posture_score":
            posture_score,
        "shoulder_alignment_score":
            shoulder_alignment_score,
        "head_position_score":
            head_position_score,
        "body_language_score":
            body_language_score
    })

    return result

In [ ]:
# Calculate Final Interview Score

def calculate_final_score(
    session_results
):

    if len(session_results) == 0:
        return 0

    return round(
        sum(
            item["overall_score"]
            for item in session_results
        )/len(session_results), 2
    )

In [ ]:
# Generate Final Report
# Gemini + Fallback

def generate_final_report(
    session_results,
    final_score
):

    timestamp = datetime.now().strftime(
        "%d-%m-%Y %H:%M"
    )

    role = session_results[0].get(
        "role",
        "N/A"
    )

    company = session_results[0].get(
        "company",
        "N/A"
    )

    strengths = []
    weaknesses = []

    question_summary = ""

    for i, result in enumerate(session_results):

        strengths.extend(
            result["strengths"]
        )

        weaknesses.extend(
            result["weaknesses"]
        )

        question_summary += f"""

Question {i+1}

Question:
{result["question"]}

Technical Score:
{result["technical_score"]}/10

Communication Score:
{result["communication_score"]}/10

Completeness Score:
{result["completeness_score"]}/10

Confidence Score:
{result["confidence_score"]}/10

Body Language Score:
{result["body_language_score"]}/10

Overall Score:
{result["overall_score"]}/10

"""

    prompt = f"""
You are an experienced HR interviewer.

Generate a professional interview report.

Interview Date:
{timestamp}

Role:
{role}

Company:
{company}

===================================

QUESTION-WISE PERFORMANCE

{question_summary}

===================================

FINAL OVERALL SCORE

{final_score}/10

===================================

COMMON STRENGTHS

{list(set(strengths))}

COMMON WEAKNESSES

{list(set(weaknesses))}

Generate a report with the following headings:

1. Overall Performance

2. Question-wise Summary

3. Technical Assessment

4. Communication Assessment

5. Behaviour Assessment

6. Key Strengths

7. Areas of Improvement

8. Final Recommendation

Keep the report under 300 words.
"""

    try:

        response = client.models.generate_content(

            model="gemini-2.5-flash",

            contents=prompt

        )

        return response.text

    except Exception:

        report = f"""
=============================
INTERVIEW REPORT
=============================

Interview Date:
{timestamp}

Role:
{role}

Company:
{company}

=============================
QUESTION-WISE PERFORMANCE
=============================

{question_summary}

=============================
FINAL RESULT
=============================

Overall Interview Score:
{final_score}/10

Strengths:

"""

        for s in sorted(set(strengths)):
            report += f"- {s}\n"

        report += "\nAreas of Improvement:\n\n"

        for w in sorted(set(weaknesses)):
            report += f"- {w}\n"

        report += """
Final Recommendation:
The candidate demonstrated good technical knowledge and communication skills throughout the interview.
Continued practice in the suggested improvement areas is recommended to further enhance interview performance.
Overall Recommendation:
The candidate demonstrated satisfactory interview performance with good technical understanding and communication skills.
Continued practice in behavioural communication and technical depth will further improve future interview performance.
Recommendation:
Suitable for further interview rounds.
"""

        return report

In [ ]:
# Save Interview Score

def save_interview_score(
    score
):

    score_history.append(
        score
    )

    return score_history

In [ ]:
# Calculate Improvement

def calculate_improvement(
    score_history
):

    if len(
        score_history
    ) < 2:

        return 0

    return round(

        score_history[-1]
        -
        score_history[0],

        2
    )

In [ ]:
# Generate Progress Graph

def generate_progress_plot():

    if len(score_history) == 0:
        return None

    plt.figure(figsize=(7,5))

    plt.plot(
        range(1, len(score_history)+1),
        score_history,
        marker="o",
        linewidth=2
    )

    for i, score in enumerate(score_history):
        plt.text(
            i + 1,
            score,
            f"{score:.2f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

    plt.title("Interview Performance Progress")
    plt.xlabel("Interview Session")
    plt.ylabel("Overall Score")
    plt.xticks(range(1, len(score_history)+1))
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(
        "progress_graph.png"
    )
    plt.close()

    return "progress_graph.png"

In [ ]:
def generate_progress_report(score_history):

    if len(score_history) == 0:

        return {
            "Total Interviews": 0,
            "Latest Score": 0,
            "Best Score": 0,
            "Average Score": 0,
            "Improvement": 0
        }

    latest = score_history[-1]
    best = max(score_history)
    average = round(
        sum(score_history) / len(score_history),
        2
    )
    improvement = round(
        latest - score_history[0],
        2
    )

    return {
        "Total Interviews": len(score_history),
        "Latest Score": latest,
        "Best Score": best,
        "Average Score": average,
        "Improvement": improvement
    }

In [ ]:
QUESTION_BANK = {

    "Google": [

        "Explain the difference between an abstract class and an interface.",

        "How does HashMap work internally in Java?",

        "What is a deadlock and how can it be prevented?",

        "Explain multithreading in Java.",

        "Difference between process and thread.",

        "What are SOLID principles?",

        "Explain JVM memory structure.",

        "What is synchronization in Java?",

        "How does garbage collection work?",

        "Explain dependency injection."
    ],

    "Amazon": [

        "What are the SOLID principles?",

        "Explain database normalization.",

        "What is a race condition?",

        "Explain REST API architecture.",

        "What is CAP theorem?",

        "Explain system design basics.",

        "What is a microservice?",

        "Difference between SQL and NoSQL.",

        "Explain load balancing.",

        "What is eventual consistency?"
    ],

    "Microsoft": [

        "What is polymorphism?",

        "Difference between TCP and UDP.",

        "What is a binary search tree?",

        "Explain synchronization.",

        "How does garbage collection work?",

        "Explain inheritance.",

        "What is encapsulation?",

        "What are design patterns?",

        "Explain DFS and BFS.",

        "What is dynamic programming?"
    ],

    "TCS": [

        "What is OOP?",

        "Difference between stack and queue.",

        "What is DBMS?",

        "Explain OS scheduling.",

        "What is inheritance?",

        "What is normalization?",

        "Explain SDLC.",

        "What is a primary key?",

        "Difference between C and Java.",

        "What is a compiler?"
    ],

    "Infosys": [

        "What is polymorphism?",

        "Difference between process and thread.",

        "Explain cloud computing.",

        "What is DBMS?",

        "What is SDLC?",

        "Explain data structures.",

        "What is exception handling?",

        "What is normalization?",

        "Difference between interface and abstract class.",

        "Explain operating systems."
    ],

    "Wipro": [

        "What is OOP?",

        "Explain DBMS.",

        "Difference between array and linked list.",

        "What is software testing?",

        "What is a constructor?",

        "Explain polymorphism.",

        "What is encapsulation?",

        "What is normalization?",

        "Explain SDLC.",

        "What is a deadlock?"
    ],

    "Accenture": [

        "Explain Agile methodology.",

        "What is cloud computing?",

        "What is a REST API?",

        "Difference between SQL and NoSQL.",

        "Explain data structures.",

        "What is exception handling?",

        "Explain OOP concepts.",

        "What is normalization?",

        "What is multithreading?",

        "Explain SDLC."
    ],

    "IBM": [

        "What is machine learning?",

        "Difference between supervised and unsupervised learning.",

        "Explain DBMS.",

        "What is cloud computing?",

        "Explain object-oriented programming.",

        "What is data preprocessing?",

        "Difference between AI and ML.",

        "Explain classification.",

        "What is normalization?",

        "Explain regression."
    ],

    "Meta": [

        "What is a graph data structure?",

        "Explain BFS and DFS.",

        "What is a distributed system?",

        "Explain caching.",

        "What is sharding?",

        "Explain load balancing.",

        "What is a message queue?",

        "Difference between SQL and NoSQL.",

        "What is eventual consistency?",

        "Explain CAP theorem."
    ],

    "Apple": [

        "Explain MVC architecture.",

        "What is memory management?",

        "What is multithreading?",

        "Difference between process and thread.",

        "Explain OOP principles.",

        "What is dependency injection?",

        "Explain design patterns.",

        "What is synchronization?",

        "What is encapsulation?",

        "Explain data structures."
    ]
}

In [ ]:
interview_session = {
    "role": "",
    "company": "",
    "questions": [],
    "current_question": 0,
    "results": [],
    "completed": False
}

def ui_generate_question(
    role,
    company,
    question_count
):

    global interview_session

    interview_session = {
        "role": role,
        "company": company,
        "questions": [],
        "current_question": 0,
        "results": [],
        "completed": False

    }

    if company in QUESTION_BANK:
        interview_session["questions"] = random.sample(
            QUESTION_BANK[company],
            min(
                question_count,
                len(
                    QUESTION_BANK[company]
                )
            )
        )

    else:
        interview_session["questions"] = get_question_list(
            role,
            company
        )[:question_count]

    return f"""
        Interview Started
        Question 1 of {len(interview_session['questions'])}
        {interview_session['questions'][0]}
    """

In [ ]:
def get_current_question():

    global interview_session

    if interview_session["completed"]:
        return None

    index = interview_session["current_question"]

    return interview_session["questions"][index]

In [ ]:
def move_to_next_question():

    global interview_session

    interview_session["current_question"] += 1

    if (
        interview_session["current_question"]
        >= len(
            interview_session["questions"]
          )
      ):
        interview_session["completed"] = True
        return None

    index = interview_session["current_question"]

    return f"""
        Question {index+1} of {len(interview_session['questions'])}
        {interview_session["questions"][index]}
    """

In [ ]:
def get_interview_status():

    global interview_session

    total = len(
        interview_session["questions"]
    )

    current = interview_session["current_question"]

    if interview_session["completed"]:
        return "Interview Completed"

    return f"Question {current+1} of {total}"

In [ ]:
# Store Question Result

def save_question_result(
    result
):

    global interview_session

    interview_session["results"].append(
        result
    )

In [ ]:
# Generate Human Behaviour Analysis Report

def generate_confidence_report(result):

    report = f"""
===============================
 HUMAN BEHAVIOUR ANALYSIS
===============================

FACIAL ANALYSIS

Face Visibility Score :
{result.get("face_visibility_score", 0):.2f}/10

Head Alignment Score :
{result.get("head_alignment_score", 0):.2f}/10

Eye Contact Score :
{result.get("eye_contact_score", 0):.2f}/10

Smile Score :
{result.get("smile_score", 0):.2f}/10


BODY LANGUAGE ANALYSIS

Posture Score :
{result.get("posture_score", 0):.2f}/10

Shoulder Alignment Score :
{result.get("shoulder_alignment_score", 0):.2f}/10

Head Position Score :
{result.get("head_position_score", 0):.2f}/10

Body Language Score :
{result.get("body_language_score", 0):.2f}/10


OVERALL BEHAVIOUR

Confidence Score :
{result.get("confidence_score", 0):.2f}/10

Engagement Score :
{result.get("engagement_score", 0):.2f}/10

"""

    confidence = result.get(
        "confidence_score",
        0
    )

    engagement = result.get(
        "engagement_score",
        0
    )

    body = result.get(
        "body_language_score",
        0
    )

    if confidence >= 8:
        report += "\nConfidence Assessment : Excellent\n"

    elif confidence >= 6:
        report += "\nConfidence Assessment : Good\n"

    else:
        report += "\nConfidence Assessment : Needs Improvement\n"

    if engagement >= 8:
        report += "Engagement Assessment : Excellent\n"

    elif engagement >= 6:
        report += "Engagement Assessment : Good\n"

    else:
        report += "Engagement Assessment : Needs Improvement\n"

    if body >= 8:
        report += "Body Language Assessment : Excellent\n"

    elif body >= 6:
        report += "Body Language Assessment : Good\n"

    else:
        report += "Body Language Assessment : Needs Improvement\n"

    return report

In [ ]:
def generate_score_dashboard(result):

    return f"""
========== INTERVIEW SCORE DASHBOARD ==========

Response Duration    : {result['response_time']} sec

Technical Score      : {result['technical_score']}/10

Communication Score  : {result['communication_score']}/10

Completeness Score   : {result['completeness_score']}/10

Confidence Score     : {result['confidence_score']}/10

Engagement Score     : {result['engagement_score']}/10

Body Language Score  :  {result['body_language_score']}/10

Overall Score        : {result['overall_score']}/10
"""

In [ ]:
def ui_evaluate_answer(
    role,
    company,
    question,
    video_file
):
    print("VIDEO:", video_file)
    print("TYPE:", type(video_file))

    global interview_session
    if video_file is None:
        raise gr.Error(
          "Please record your answer before submitting."
        )

    if isinstance(video_file, dict):
        video_file = video_file.get("path")

    # Convert Video To Transcript

    transcript = transcribe_video(
        video_file
    )

    if len(transcript.strip()) < 10:
        raise gr.Error(
          "Speech not detected. Please answer the interview question."
        )

    current_question = get_current_question()

    # Evaluate Current Question

    result = process_answer(
        current_question,
        transcript,
        video_file
    )

    result["response_time"] = get_video_duration(
        video_file
    )

    result["role"] = role
    result["company"] = company

    save_question_result(
        result
    )

    current_number = len(interview_session["results"])

    total_questions = len(interview_session["questions"])

    save_interview_score(
        result["overall_score"]
    )

    score_dashboard = generate_score_dashboard(
        result
    )

    confidence_report = generate_confidence_report(
        result
    )

    next_question = move_to_next_question()

    # Interview Finished

    if next_question is None:
        final_score = calculate_final_score(
            interview_session["results"]
        )

        report = generate_final_report(
            interview_session["results"],
            final_score
        )

        progress = generate_progress_report(
            score_history
        )

        progress["Questions Answered"] = len(
            interview_session["results"]
        )

        progress["Interview Role"] = role

        progress["Target Company"] = company

        progress_graph = generate_progress_plot()

        question_text = f"""
          Interview Completed
          Total Questions:
          {len(interview_session["questions"])}
          Overall Score:
          {final_score}/10
        """
    else:
        report = f"""
          Question Evaluated Successfully
          Progress:
          {current_number} of {total_questions} Questions Completed
          Record your answer for the next question.
        """
        progress = {
          "Questions Completed":
              current_number,
          "Remaining":
              total_questions - current_number
        }
        progress_graph = None
        question_text = next_question

    return (
        question_text,
        transcript,
        result,
        score_dashboard,
        confidence_report,
        report,
        progress,
        progress_graph
    )

In [ ]:
with gr.Blocks() as demo:

    gr.Markdown(
        "# AI-Powered Interview Preparation Platform"
    )

    role = gr.Dropdown(

        choices=[
            "Software Engineer",
            "Data Scientist",
            "Machine Learning Engineer",
            "Backend Developer",
            "Frontend Developer",
            "Full Stack Developer",
            "DevOps Engineer",
            "Cybersecurity Analyst",
            "Business Analyst",
            "Product Manager"
        ],

        value="Software Engineer",
        label="Target Role"
    )

    company = gr.Dropdown(

        choices=[
            "Google",
            "Amazon",
            "Microsoft",
            "TCS",
            "Infosys",
            "Wipro",
            "Accenture",
            "IBM",
            "Meta",
            "Apple"
        ],

        value="Google",
        label="Target Company"
    )

    question_count = gr.Dropdown(
        choices=[1,2,3,4,5],
        value=3,
        label="Number of Questions"
    )

    generate_btn = gr.Button(
        "Start Interview"
    )

    question = gr.Textbox(
        label="Current Interview Question",
        lines=8
    )

    generate_btn.click(

        fn=ui_generate_question,

        inputs=[
            role,
            company,
            question_count
        ],

        outputs=question
    )

    video_input = gr.Video(
        sources=["upload"],
        format="mp4",
        #include_audio=True,
        #interactive=True,
        label="Upload Video"
    )

    evaluate_btn = gr.Button(
        "Submit Answer"
    )

    transcript = gr.Textbox(
        label="Candidate Transcript",
        lines=6
    )

    result_json = gr.JSON(
        label="Technical Evaluation"
    )

    score_dashboard = gr.Textbox(
        label="Interview Score Dashboard",
        lines=10
    )

    confidence_report = gr.Textbox(
        label="Human Behaviour Analysis",
        lines=12
    )

    report = gr.Textbox(
        label="Final Report",
        lines=12
    )

    progress_report = gr.JSON(
        label="Progress Tracking"
    )

    progress_graph = gr.Image(
        label="Interview Progress Graph"
    )

    evaluate_btn.click(
      fn=ui_evaluate_answer,

      inputs=[
        role,
        company,
        question,
        video_input
      ],

      outputs=[
        question,
        transcript,
        result_json,
        score_dashboard,
        confidence_report,
        report,
        progress_report,
        progress_graph
      ]
    )

In [ ]:
demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5451e3078916bd609d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


VIDEO: /tmp/gradio/7fffd3b14f6a651199d491c34989142d3e881d316fb230a13e378833c99ff8dd/WIN_20260629_21_44_51_Pro.mp4
TYPE: <class 'str'>


  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



Using Gemini
VIDEO: /tmp/gradio/39b9732cba0d8a359ce3d672c1d40461159de1d79e3ea4a7337bef487be560a1/WIN_20260629_21_48_58_Pro.mp4
TYPE: <class 'str'>


  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



Using Gemini
VIDEO: /tmp/gradio/90ebd1d7386d6dbeb05c515f8237758017c04f26cddeb72a23626189be0eac04/WIN_20260629_21_51_58_Pro.mp4
TYPE: <class 'str'>


  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



Using Gemini
VIDEO: None
TYPE: <class 'NoneType'>


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1698, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 63, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^